In [1]:
# general imports
import pandas as pd
import os
import json

In [ ]:
# Optional environment setup.
# Set these before initializing vLLM, PyTorch, sentence-transformers, or transformers models.

# Restrict this notebook process to specific GPUs.
# Examples: "0" for one GPU, "0,1" for two GPUs.
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Optional: set for higher Hugging Face Hub rate limits and faster first-time downloads.
# os.environ["HF_TOKEN"] = "HF..."

# MMAI Examples

## 1. Trial summarization walkthrough


### Read in data

In [2]:
trials_path = "data/scheduled__2025-09-04T230000+0000.trials_for_summarize.csv"
trials = pd.read_csv(trials_path)
trials = trials.iloc[0:10,]

### Local mode

`summarize_trials(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [ ]:
from matchminer_ai.trials import summarize_trials

trial_results, metadata, qc_report = summarize_trials(
    trials, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()

/home/sabrina/mmai-package/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 06-29 19:15:37 [utils.py:278] non-default args: {'max_model_len': 30000, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'language_model_only': True, 'model': 'google/gemma-4-31B-it'}
INFO 06-29 19:15:37 [model.py:617] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-29 19:15:37 [model.py:1752] Using max model len 30000
INFO 06-29 19:15:42 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-29 19:15:42 [config.py:100] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-29 19:15:42 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-29 19:15:42 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 06-29 19:15:42 [cuda.py:243] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attention.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
(EngineCore pid=62434) Process EngineCore:
(EngineCore pid=62434) Traceback (most recent call last):
(EngineCore pid=62434)   File "/home/sabrina/.local/share/uv/python/cpython-3.13.13-linux-x86_64-gnu/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
(EngineCore pid=62434)     self.run()
(EngineCore pid=62434)     ~~~~~~~~^^
(EngineCore pid=62434)   File "/home/sabrina/.local/share/uv/python/cpython-3.13.13-linux-x86_64-gnu/lib/python3.13/multiprocessing/process.py", line 108, in run
(EngineCore pid=62434)     self._target(*self._args, **self._kwargs)
(EngineCore pid=62434)     ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=62434)   File "/home/sabrina/mmai-package/.venv/lib/python3.13/site-packages/vllm/v1/engine/core.py", line 1139, in run_engine_core
(EngineCore pid=62434)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=62434)   File "/ho

### Remote mode

Run this cell instead of the local-mode cell if you want to send trial summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values. If your endpoint exposes the model under an API alias, set `config.remote["served_model_names"]["trial"]` to that served name.

In [ ]:
from matchminer_ai import load_preset
from matchminer_ai.trials import summarize_trials

os.environ["OPENAI_API_KEY"] = "not-needed"

# Optional: start a local vLLM server if you do not already have one.
# Optional: choose GPUs for the server process before starting it.
# from matchminer_ai.llm.vllm_server import start_vllm_server
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# process = start_vllm_server(task="trial")

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]
# Optional: use this when the endpoint request model is an API alias.
# config.remote["served_model_names"]["trial"] = "your-served-model-name"
# config.remote["send_vllm_extra_body"] = False  # for non-vLLM OpenAI-compatible APIs

trial_results, metadata, qc_report = summarize_trials(
    trials, config=config, return_metadata=True, return_qc=True
)

# Optional: stop the local vLLM server if you started it above.
# process.terminate()

### View results

In [4]:
trial_results.head()

,trial_id,clinical_space_number,clinical_space_summary,general_exclusion_criteria,space_trial_id
0,NCT03319901,1,Age range allowed: >= 60 Years. Sex allowed: B...,Poor performance status (eg ECOG > 2)\nHepatic...,NCT03319901-1
1,NCT03319901,2,Age range allowed: >= 60 Years. Sex allowed: B...,Poor performance status (eg ECOG > 2)\nHepatic...,NCT03319901-2
2,NCT03319901,3,Age range allowed: >= 18 Years. Sex allowed: B...,Poor performance status (eg ECOG > 2)\nHepatic...,NCT03319901-3
3,NCT03319901,4,Age range allowed: >= 18 Years. Sex allowed: B...,Poor performance status (eg ECOG > 2)\nHepatic...,NCT03319901-4
4,NCT04792489,1,Age range allowed: 18 years or older. Sex allo...,"Poor performance status (ECOG greater than 1, ...",NCT04792489-1


Metadata contains a snapshot of the config used and metadata on the model used. 

In [5]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 100000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'llm_match_quality': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True},
   'llm_exclusion_criteria': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'served_model_names': {'trial': 'google/gemma-4-31B-it',
    'patient': 'google/gemma-4-31B-it',
    'llm_match_quality': 'google/gemma-4-31B-it',
    'llm_exclusion_criteria': 'google/gemma-4-31B-it'},
   's

QC report on the trial summarization run

In [6]:
qc_report

,metric,value,denominator,percent,ids
0,trials_missing_in_output,0.0,10.0,0.0,[]
1,trials_truncated_llm_response,0.0,10.0,0.0,[]
2,spaces_exceed_embedding_token_limit,0.0,30.0,0.0,[]
3,spaces_per_trial_min,1.0,NaN,NaN,[]
4,spaces_per_trial_median,2.5,NaN,NaN,[]
5,spaces_per_trial_max,9.0,NaN,NaN,[]
6,trials_with_non_distinct_spaces,0.0,10.0,0.0,[]
7,spaces_dropped_missing_keyword:Age,0.0,30.0,0.0,[]
8,spaces_dropped_missing_keyword:Sex,0.0,30.0,0.0,[]
9,spaces_dropped_missing_keyword:Cancer type,0.0,30.0,0.0,[]


Save checked-in trial example artifacts


In [7]:
trial_results.to_csv("output/trial_summaries.csv", index=None)
with open("output/trial_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/trial_qc_report.csv", index=None)

## 2. Patient summarization walkthrough


### Read in initial notes

In [3]:
notes_path = "data/note_set_1.csv"
notes = pd.read_csv(notes_path)

### Local mode

`summarize_patients(...)` uses the local in-memory vLLM backend by default. Run this cell if you want local mode.


In [12]:
from matchminer_ai.patients import summarize_patients

patient_summaries, metadata, qc_report = summarize_patients(
    notes, return_metadata=True, return_qc=True
)

# Optional: clear the cached local vLLM engine if GPU memory is tight.
# from matchminer_ai.llm.backends import clear_local_llm_cache
# clear_local_llm_cache()

/home/sabrina/mmai-package/src/matchminer_ai/patients/prepare.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  normalized["note_date"] = pd.to_datetime(normalized["note_date"])


INFO 06-29 18:02:54 [utils.py:278] non-default args: {'max_model_len': 34208, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'language_model_only': True, 'model': 'google/gemma-4-31B-it'}
INFO 06-29 18:02:54 [model.py:617] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-29 18:02:54 [model.py:1752] Using max model len 34208
INFO 06-29 18:02:54 [config.py:100] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-29 18:02:54 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 06-29 18:02:54 [cuda.py:243] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attention.
(EngineCore pid=44319) INFO 06-29 18:03:03 [core.py:112] Initializing a V1 LLM engine (v0.22.0) with config: model='google/gemma-4-31B-it', speculative_config=None, token

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:12<00:12, 12.67s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  7.38s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  8.17s/it]
(EngineCore pid=44319) 


(EngineCore pid=44319) INFO 06-29 18:03:24 [default_loader.py:397] Loading weights took 16.43 seconds
(EngineCore pid=44319) INFO 06-29 18:03:25 [gpu_model_runner.py:5132] Model loading took 57.91 GiB memory and 17.785029 seconds
(EngineCore pid=44319) INFO 06-29 18:03:33 [backends.py:1089] Using cache directory: /home/sabrina/.cache/vllm/torch_compile_cache/76884c0395/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=44319) INFO 06-29 18:03:33 [backends.py:1148] Dynamo bytecode transform time: 7.56 s
(EngineCore pid=44319) INFO 06-29 18:03:46 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 11.820 s
(EngineCore pid=44319) INFO 06-29 18:03:46 [decorators.py:311] Directly load AOT compilation from path /home/sabrina/.cache/vllm/torch_compile_cache/torch_aot_compile/f70134b0046169638aed928c4357c3f46ef02ab7c2e1ee3e804b854ad439a1ef/rank_0_0/model
(EngineCore pid=44319) INFO 06-29 18:03:46 [monitor.py:53] torch.compile took 20.1

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:07<00:00,  6.75it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:04<00:00,  7.97it/s]


...

(EngineCore pid=44319) WARNING 06-29 18:04:09 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Rendering prompts:  17%|█▋        | 2/12 [00:00<00:00, 14.83it/s]

(EngineCore pid=44319) WARNING 06-29 18:04:09 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 12/12 [15:27<00:00, 77.28s/it, est. speed input: 220.89 toks/s, output: 35.74 toks/s]


(EngineCore pid=44319) INFO 06-29 18:47:06 [core.py:1266] Shutdown initiated (timeout=0)
(EngineCore pid=44319) INFO 06-29 18:47:06 [core.py:1289] Shutdown complete


### Remote mode

Run this cell instead of the local-mode cell if you want to send patient summarization requests to an existing OpenAI-compatible endpoint. If you do not already have a vLLM server running, `start_vllm_server(...)` can start one from the config values. If your endpoint exposes the model under an API alias, set `config.remote["served_model_names"]["patient"]` to that served name.

In [ ]:
from matchminer_ai import load_preset
from matchminer_ai.patients import summarize_patients

os.environ["OPENAI_API_KEY"] = "not-needed"

config = load_preset("default")
config.remote["enabled"] = True
config.remote["server_urls"] = ["http://localhost:8000/v1"]
# Optional: use this when the endpoint request model is an API alias.
# config.remote["served_model_names"]["patient"] = "your-served-model-name"
# config.remote["send_vllm_extra_body"] = False  # for non-vLLM OpenAI-compatible APIs

# Optional: try a quantized Gemma 4 server with speculative decoding for faster
# patient summarization. Use the same config for starting the server and running
# summarization so prompt construction and remote requests stay consistent.
# config.patient["model_name"] = "google/gemma-4-31B-it-qat-w4a16-ct"
# config.patient["reasoning_parser"] = "gemma4"
# config.local["patient"]["max_model_len"] = 32768
# config.local["patient"]["tensor_parallel_size"] = 2
# config.local["patient"]["gpu_memory_utilization"] = 0.40
# qat_extra_args = [
#     "--limit-mm-per-prompt",
#     '{"image": 0, "audio": 0}',
#     "--speculative-config",
#     '{"method": "mtp", "model": "google/gemma-4-31B-it-qat-q4_0-unquantized-assistant", "num_speculative_tokens": 6}',
# ]

# Optional: start a local vLLM server if you do not already have one.
# Optional: choose GPUs for the server process before starting it.
# from matchminer_ai.llm.vllm_server import start_vllm_server
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# process = start_vllm_server(
#     config=config,
#     task="patient",
#     extra_args=qat_extra_args,  # omit this argument when using the default model
# )

patient_summaries, metadata, qc_report = summarize_patients(
    notes, config=config, return_metadata=True, return_qc=True
)

# Optional: stop the local vLLM server if you started it above.
# process.terminate()



### Updating existing patient summaries

Run this optional cell after the local- or remote-mode cell above if you want to simulate a longitudinal update. It reads a second set of new notes, uses the existing `patient_summaries` as the starting point, then updates those summaries with the new notes.


In [13]:
from matchminer_ai.patients import summarize_patients

new_notes_path = "data/note_set_2.csv"
new_notes = pd.read_csv(new_notes_path)

existing_summaries = patient_summaries.assign(
    patient_summary=lambda df: (
        df["cancer_history_summary"].fillna("")
        + "\n\nBoilerplate:\n"
        + df["general_exclusion_criteria_evidence"].fillna("")
    )
)[["patient_id", "patient_summary"]]

patient_summaries, metadata, qc_report = summarize_patients(
    new_notes,
    existing_summaries=existing_summaries,
    return_metadata=True,
    return_qc=True,
)

Processed prompts: 100%|██████████| 12/12 [10:43<00:00, 53.64s/it, est. speed input: 220.32 toks/s, output: 54.59 toks/s]


### View results

In [14]:
patient_summaries.head()

,patient_id,last_note_date,cancer_history_summary,general_exclusion_criteria_evidence
0,11609,2026-05-03,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...
1,12019,2026-05-14,Age: 52\nSex: Female\nCancer type: Breast canc...,"Hypertension, hyperlipidemia, Grade 2 ribocicl..."
2,15687,2025-10-07,Age: 68\nSex: Male\nCancer type: Lung cancer\n...,ECOG 1-2. Hypertension. Hyperlipidemia. Osteop...
3,16916000000,2024-11-14,Age: 21\nSex: Male\nCancer type: Osteosarcoma\...,ECOG 0-1. History of Grade 2 immune-mediated c...
4,1722,2025-02-10,Age: 64\nSex: Male\nCancer type: Gastro-esopha...,ECOG 1. Hypertension. Hyperlipidemia. Former s...


Metadata contains a snapshot of the config used and metadata on the models used. 

In [15]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'llm_match_quality': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True},
   'llm_exclusion_criteria': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'served_model_names': {'trial': 'google/gemma-4-31B-it',
    'patient': 'google/gemma-4-31B-it',
    'llm_match_quality': 'google/gemma-4-31B-it',
    'llm_exclusion_criteria': 'google/gemma-4-31B-it'},
   'se

QC report on patient summarization

In [16]:
qc_report

,metric,value,denominator,percent,ids
0,patients_dropped_noninformative_summary,0,12,0.0,[]
1,patients_exceed_embedding_token_limit,0,12,0.0,[]
2,patients_exclusion_criteria_not_extracted,0,12,0.0,[]
3,patients_missing_keyword:Age,0,12,0.0,[]
4,patients_missing_keyword:Sex,0,12,0.0,[]
5,patients_missing_keyword:Cancer type,0,12,0.0,[]
6,patients_missing_keyword:Histology,0,12,0.0,[]
7,patients_missing_keyword:Current extent,0,12,0.0,[]
8,patients_missing_keyword:Biomarkers,0,12,0.0,[]
9,patients_missing_keyword:Treatment history,0,12,0.0,[]


Save checked-in patient example artifacts

In [17]:
patient_summaries.to_csv("output/patient_summaries.csv", index=None)
with open("output/patient_summarization_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
qc_report.to_csv("output/patient_qc_report.csv", index=None)

## 3. Embed summarized entities for matching


In [18]:
from matchminer_ai.embedding import embed_for_matching

In [ ]:
# Uncomment if you want to start from previously generated summaries
# trial_results = pd.read_csv("output/trial_summaries.csv")
# patient_summaries = pd.read_csv("output/patient_summaries.csv", dtype="str")

### Trials

In [19]:
embedded_trials, metadata = embed_for_matching(
    trial_results, entity_type="trial", return_metadata=True
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 872.11it/s]


In [20]:
embedded_trials.head()

,space_trial_id,embedding
0,NCT03319901-1,"[-0.024169921875, -0.041748046875, 0.004364013..."
1,NCT03319901-2,"[0.0106201171875, 0.008056640625, 0.0078125, 0..."
2,NCT03319901-3,"[-0.003021240234375, -0.0142822265625, 0.00805..."
3,NCT03319901-4,"[0.03857421875, 0.04443359375, 0.011962890625,..."
4,NCT04792489-1,"[-0.025390625, -0.03466796875, 0.0046691894531..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [21]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'llm_match_quality': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True},
   'llm_exclusion_criteria': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'served_model_names': {'trial': 'google/gemma-4-31B-it',
    'patient': 'google/gemma-4-31B-it',
    'llm_match_quality': 'google/gemma-4-31B-it',
    'llm_exclusion_criteria': 'google/gemma-4-31B-it'},
   'se

### Patients

In [22]:
embedded_patients, metadata = embed_for_matching(
    patient_summaries, entity_type="patient", return_metadata=True
)

In [23]:
embedded_patients.head()

,patient_id,embedding
0,11609,"[0.004150390625, 0.0103759765625, -0.004791259..."
1,12019,"[-0.0546875, 0.060546875, -0.0078125, 0.034667..."
2,15687,"[0.01611328125, -0.006378173828125, -0.0050964..."
3,16916000000,"[0.06298828125, -0.00194549560546875, -0.00424..."
4,1722,"[0.0189208984375, 0.051513671875, -0.003967285..."


Metadata contains a snapshot of the config used and metadata on the models used. 

In [24]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'llm_match_quality': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True},
   'llm_exclusion_criteria': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'served_model_names': {'trial': 'google/gemma-4-31B-it',
    'patient': 'google/gemma-4-31B-it',
    'llm_match_quality': 'google/gemma-4-31B-it',
    'llm_exclusion_criteria': 'google/gemma-4-31B-it'},
   'se

In [25]:
# Optional local export if you want to reuse embeddings without rerunning this step.
embedded_patients.to_parquet("output/embedded_patients.parquet")
embedded_trials.to_parquet("output/embedded_trials.parquet")

## 4. Generate candidate matches

Generate ranked trial-space candidates for each patient.


In [5]:
from matchminer_ai.matching import generate_candidate_matches

In [ ]:
# Uncomment if you saved embeddings locally and want to start from them
# embedded_patients = pd.read_parquet('output/embedded_patients.parquet')
# embedded_trials = pd.read_parquet('output/embedded_trials.parquet')

### Candidate matches


In [7]:
candidate_matches = generate_candidate_matches(
    query_df=embedded_patients, corpus_df=embedded_trials
)

In [8]:
candidate_matches.head()

,patient_id,space_trial_id,similarity_score,rank
0,11609,NCT05785741-1,0.388877,1
1,11609,NCT05538130-1,0.355187,2
2,11609,NCT06234423-2,0.335172,3
3,11609,NCT05538130-3,0.307204,4
4,11609,NCT04792489-6,0.196194,5


## 5. Match quality check


### Classifier-based match-quality check

Run the trained classifier checker after adding patient and trial summary context.

In [10]:
from matchminer_ai.matching import score_match_quality

Add back the patient and trial summary context required by the match-quality checker.


In [11]:
candidate_matches_with_context = (
    candidate_matches.merge(
        patient_summaries[["patient_id", "cancer_history_summary"]],
        on="patient_id",
        how="left",
    ).merge(
        trial_results[["space_trial_id", "clinical_space_summary"]],
        on="space_trial_id",
        how="left",
    )
)
candidate_matches_with_context.head()

,patient_id,space_trial_id,similarity_score,rank,cancer_history_summary,clinical_space_summary
0,11609,NCT05785741-1,0.388877,1,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: 18 years or older. Sex allo...
1,11609,NCT05538130-1,0.355187,2,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: NA. Sex allowed: Both. Canc...
2,11609,NCT06234423-2,0.335172,3,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: >=18 years. Sex allowed: Ma...
3,11609,NCT05538130-3,0.307204,4,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: NA. Sex allowed: Both. Canc...
4,11609,NCT04792489-6,0.196194,5,Age: 71\nSex: Female\nCancer type: Glioblastom...,Age range allowed: 18 years or older. Sex allo...


Run the match-quality checker.


In [12]:
match_quality_matches, metadata = score_match_quality(
    candidate_matches_with_context, return_metadata=True
)

/home/sabrina/mmai-package/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 174/174 [00:00<00:00, 5442.27it/s]


In [13]:
match_quality_matches.head()

,patient_id,space_trial_id,match_quality_score,match_quality_pass
0,11609,NCT05785741-1,0.551904,True
1,11609,NCT05538130-1,0.477283,True
2,11609,NCT06234423-2,0.439670,True
3,12019,NCT05608252-1,0.998419,True
4,12019,NCT05538130-1,0.397200,True


In [14]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'llm_match_quality': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True},
   'llm_exclusion_criteria': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'served_model_names': {'trial': 'google/gemma-4-31B-it',
    'patient': 'google/gemma-4-31B-it',
    'llm_match_quality': 'google/gemma-4-31B-it',
    'llm_exclusion_criteria': 'google/gemma-4-31B-it'},
   'se

### LLM-based match-quality check

This optional checker uses the configured LLM backend instead of the classifier checker. Debug mode includes the raw response columns, which are useful when reviewing parsed scores.

In [38]:
from matchminer_ai import load_preset
from matchminer_ai.matching import score_match_quality_with_llm

llm_checker_config = load_preset("default")
llm_checker_config.debug_mode = True

llm_match_quality_matches, llm_match_quality_metadata = score_match_quality_with_llm(
    candidate_matches_with_context,
    config=llm_checker_config,
    return_metadata=True,
)


Processed prompts: 100%|██████████| 240/240 [13:48<00:00,  3.45s/it, est. speed input: 406.13 toks/s, output: 214.51 toks/s]


In [40]:
llm_match_quality_matches.head()

,patient_id,space_trial_id,llm_match_quality_score,llm_match_quality_verdict,llm_match_quality_response,llm_match_quality_reasoning
0,11609,NCT05785741-1,2,Score:2,The patient is a 71-year-old female with progr...,* Age: $\ge$ 18 years.\n * Sex: Male or...
1,11609,NCT05538130-1,2,Score:2,"The patient has glioblastoma, which is a solid...",* Age Range: NA (No restriction)\n * Se...
2,11609,NCT06234423-2,2,Score:2,The patient is a 71-year-old female with advan...,* Age Range: $\ge 18$ years.\n * Sex: M...
3,11609,NCT05538130-3,0,Score:0,The clinical trial requires a specific biomark...,* Age Range: NA (Patient is 71 - OK)\n * ...
4,11609,NCT04792489-6,0,Score:0,The clinical trial is designed specifically fo...,* Age Range: $\ge 18$ years.\n * Sex: B...


## 6. Exclusion criteria check


### Classifier-based exclusion criteria check

Run the trained classifier checker after adding patient and trial exclusion-criteria context.

In [15]:
# limit to unique patient-trial ID pairs
match_quality_matches["trial_id"] = (
    match_quality_matches["space_trial_id"].str.split("-").str[0]
)
unique_patient_trial_pairs = match_quality_matches[
    ["patient_id", "trial_id"]
].drop_duplicates()

# add in exclusion criteria for patients and trials
patient_trial_pairs_with_exclusion_context = unique_patient_trial_pairs.merge(
    patient_summaries[["patient_id", "general_exclusion_criteria_evidence"]],
    on="patient_id",
    how="left",
).merge(
    trial_results[["trial_id", "general_exclusion_criteria"]].drop_duplicates(),
    on="trial_id",
    how="left",
)
patient_trial_pairs_with_exclusion_context.head()

,patient_id,trial_id,general_exclusion_criteria_evidence,general_exclusion_criteria
0,11609,NCT05785741,ECOG 4. Essential hypertension (grade 3 during...,Poor performance status (eg ECOG >1).\nLeft ve...
1,11609,NCT05538130,ECOG 4. Essential hypertension (grade 3 during...,Brain metastasis larger than 4 cm\nHistory or ...
2,11609,NCT06234423,ECOG 4. Essential hypertension (grade 3 during...,Poor performance status (ECOG > 1).\nActive or...
3,12019,NCT05608252,"Hypertension, hyperlipidemia, Grade 2 ribocicl...",Poor performance status (ECOG >= 2).\nActive b...
4,12019,NCT05538130,"Hypertension, hyperlipidemia, Grade 2 ribocicl...",Brain metastasis larger than 4 cm\nHistory or ...


In [45]:
from matchminer_ai.matching import exclusion_criteria_check

# run exclusion criteria check
exclusion_results, metadata = exclusion_criteria_check(
    patient_trial_pairs_with_exclusion_context, return_metadata=True
)
exclusion_results.head()

Loading weights: 100%|██████████| 174/174 [00:00<00:00, 4118.35it/s]


,patient_id,trial_id,exclusion_score,exclusion_criteria_pass
0,11609,NCT05785741,0.999995,False
1,11609,NCT05538130,0.999763,True
2,11609,NCT06234423,0.999997,False
3,12019,NCT05608252,0.999766,True
4,12019,NCT05538130,0.999989,True


In [46]:
exclusion_results["exclusion_criteria_pass"].value_counts()

exclusion_criteria_pass
True     30
False     5
Name: count, dtype: int64

In [47]:
metadata

{'config_snapshot': {'version': 0,
  'debug_mode': False,
  'model_metadata_cache_dir': '.mmai_cache/model_metadata',
  'local': {'trial': {'max_model_len': 30000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'patient': {'max_model_len': 34208,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.9,
    'language_model_only': True},
   'llm_match_quality': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True},
   'llm_exclusion_criteria': {'max_model_len': 50000,
    'tensor_parallel_size': 1,
    'gpu_memory_utilization': 0.95,
    'language_model_only': True}},
  'remote': {'enabled': False,
   'server_urls': ['http://localhost:8000/v1'],
   'served_model_names': {'trial': 'google/gemma-4-31B-it',
    'patient': 'google/gemma-4-31B-it',
    'llm_match_quality': 'google/gemma-4-31B-it',
    'llm_exclusion_criteria': 'google/gemma-4-31B-it'},
   'se

### LLM-based exclusion criteria check

This optional checker uses the configured LLM backend instead of the classifier checker and returns whether each patient-trial pair passes exclusion criteria.

In [16]:
from matchminer_ai.llm.backends import clear_local_llm_cache
clear_local_llm_cache()

In [17]:
from matchminer_ai import load_preset
from matchminer_ai.matching import exclusion_criteria_check_with_llm

llm_checker_config = load_preset("default")
llm_checker_config.debug_mode = True

llm_exclusion_results, llm_exclusion_metadata = exclusion_criteria_check_with_llm(
    patient_trial_pairs_with_exclusion_context,
    config=llm_checker_config,
    return_metadata=True,
)

INFO 06-29 19:19:45 [utils.py:278] non-default args: {'max_model_len': 50000, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'language_model_only': True, 'model': 'google/gemma-4-31B-it'}
INFO 06-29 19:19:46 [model.py:617] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-29 19:19:46 [model.py:1752] Using max model len 50000
INFO 06-29 19:19:51 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-29 19:19:51 [config.py:100] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-29 19:19:51 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-29 19:19:51 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 06-29 19:19:51 [cuda.py:243] Forcing --disable_chunked_mm_input for models with multimodal-bidirectional attention.

(EngineCore pid=64331) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=64331) INFO 06-29 19:20:11 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=64331) INFO 06-29 19:20:12 [parallel_state.py:1422] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://10.128.0.42:50109 backend=nccl
(EngineCore pid=64331) INFO 06-29 19:20:12 [parallel_state.py:1735] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=64331) INFO 06-29 19:20:13 [topk_topp_sampler.py:45] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=64331) INFO 06-29 19:20:13 [gpu_model_runner.py:5037] Starting to load model google/gemma-4-31B-it...
(EngineCore pid=64331) INFO 06-29 19:20:13 [vllm.py:977] Asynchronous scheduling is enabled.
(EngineCore pid=64331) INFO 06-29 19:20:13 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:12<00:12, 12.66s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  7.36s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:16<00:00,  8.16s/it]
(EngineCore pid=64331) 


(EngineCore pid=64331) INFO 06-29 19:20:31 [default_loader.py:397] Loading weights took 16.39 seconds
(EngineCore pid=64331) INFO 06-29 19:20:32 [gpu_model_runner.py:5132] Model loading took 57.91 GiB memory and 17.504368 seconds
(EngineCore pid=64331) INFO 06-29 19:20:39 [backends.py:1089] Using cache directory: /home/sabrina/.cache/vllm/torch_compile_cache/43d3a514d4/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=64331) INFO 06-29 19:20:39 [backends.py:1148] Dynamo bytecode transform time: 6.97 s
(EngineCore pid=64331) INFO 06-29 19:20:43 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.703 s
(EngineCore pid=64331) INFO 06-29 19:20:43 [decorators.py:311] Directly load AOT compilation from path /home/sabrina/.cache/vllm/torch_compile_cache/torch_aot_compile/78cdf589f6d2d203da5a188bd914d969bba62f70816af1655edf763f177c4a3d/rank_0_0/model
(EngineCore pid=64331) INFO 06-29 19:20:43 [monitor.py:53] torch.compile took 10.44

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:07<00:00,  6.96it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:04<00:00,  8.29it/s]


(EngineCore pid=64331) INFO 06-29 19:21:02 [gpu_model_runner.py:6456] Graph capturing finished in 13 secs, took 0.82 GiB
(EngineCore pid=64331) INFO 06-29 19:21:02 [gpu_worker.py:619] CUDA graph pool memory: 0.82 GiB (actual), 0.84 GiB (estimated), difference: 0.02 GiB (2.1%).
(EngineCore pid=64331) INFO 06-29 19:21:02 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=64331) INFO 06-29 19:21:02 [core.py:302] init engine (profile, create kv cache, warmup model) took 30.06 s (compilation: 10.44 s)
(EngineCore pid=64331) INFO 06-29 19:21:03 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


Rendering prompts: 100%|██████████| 35/35 [00:00<00:00, 299.22it/s]


(EngineCore pid=64331) WARNING 06-29 19:21:03 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.
(EngineCore pid=64331) WARNING 06-29 19:21:03 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts: 100%|██████████| 35/35 [04:32<00:00,  7.78s/it, est. speed input: 101.53 toks/s, output: 211.90 toks/s]


In [18]:
llm_exclusion_results.head()

,patient_id,trial_id,llm_exclusion_criteria_pass,llm_exclusion_criteria_verdict,llm_exclusion_criteria_response,llm_exclusion_criteria_reasoning
0,11609,NCT05785741,False,YES,1. Poor performance status (eg ECOG >1): The p...,* Role: Brilliant oncologist.\n * Task:...
1,11609,NCT05538130,True,NO,1. Brain metastasis larger than 4 cm: The pati...,* Role: Brilliant oncologist.\n * Task:...
2,11609,NCT06234423,False,YES,1. Poor performance status (ECOG > 1): The pat...,* Role: Brilliant oncologist.\n * Task:...
3,12019,NCT05608252,True,NO,1. Poor performance status (ECOG >= 2): The pa...,* Role: Brilliant oncologist.\n * Task:...
4,12019,NCT05538130,True,NO,1. Brain metastasis larger than 4 cm: The pati...,* Role: Brilliant oncologist.\n * Task:...


## 7. Save final matches

Create one review table with one row per patient/trial pair. Each pair is represented by its top-scoring clinical space, rows are ranked by that top match-quality score, and exclusion-check results are included as labels without filtering rows.


In [63]:
patient_context_cols = [
    col
    for col in [
        "patient_id",
        "cancer_history_summary",
        "general_exclusion_criteria_evidence",
    ]
    if col in patient_summaries.columns
]
trial_context_cols = [
    col
    for col in [
        "space_trial_id",
        "clinical_space_number",
        "clinical_space_summary",
        "general_exclusion_criteria",
    ]
    if col in trial_results.columns
]
original_trial_cols = [
    col
    for col in [
        "trial_id",
        "oncore_id",
        "trial_title",
        "brief_summary",
        "detailed_summary",
        "eligibility_criteria",
    ]
    if col in trials.columns
]

final_matches = (
    match_quality_matches.merge(
        candidate_matches[["patient_id", "space_trial_id", "similarity_score", "rank"]],
        on=["patient_id", "space_trial_id"],
        how="left",
    )
    .merge(
        exclusion_results,
        on=["patient_id", "trial_id"],
        how="left",
    )
    .merge(
        patient_summaries[patient_context_cols],
        on="patient_id",
        how="left",
    )
    .merge(
        trial_results[trial_context_cols].drop_duplicates("space_trial_id"),
        on="space_trial_id",
        how="left",
    )
    .merge(
        trials[original_trial_cols].drop_duplicates("trial_id"),
        on="trial_id",
        how="left",
    )
)

# Keep one row per patient/trial pair, represented by the top-scoring clinical space.
final_matches = (
    final_matches.sort_values(
        ["patient_id", "trial_id", "match_quality_score"],
        ascending=[True, True, False],
    )
    .drop_duplicates(["patient_id", "trial_id"], keep="first")
    .sort_values(["patient_id", "match_quality_score"], ascending=[True, False])
    .reset_index(drop=True)
)
final_matches["match_quality_rank"] = final_matches.groupby("patient_id").cumcount() + 1

final_matches = final_matches.rename(
    columns={
        "match_quality_rank": "match_rank",
        "exclusion_criteria_pass": "passes_exclusion_criteria",
        "cancer_history_summary": "patient_summary",
        "general_exclusion_criteria_evidence": "patient_exclusion_evidence",
        "brief_summary": "trial_brief_summary",
        "detailed_summary": "trial_detailed_summary",
        "eligibility_criteria": "trial_eligibility_criteria",
        "clinical_space_summary": "matched_clinical_space_summary",
        "general_exclusion_criteria": "extracted_trial_exclusion_criteria",
    }
)

ordered_cols = [
    "patient_id",
    "trial_id",
    "match_rank",
    "passes_exclusion_criteria",
    "patient_summary",
    "patient_exclusion_evidence",
    "trial_title",
    "trial_brief_summary",
    "trial_eligibility_criteria",
    "clinical_space_number",
    "matched_clinical_space_summary",
    "extracted_trial_exclusion_criteria",
]
final_matches = final_matches[[col for col in ordered_cols if col in final_matches.columns]]
final_matches.head()


,patient_id,trial_id,match_rank,passes_exclusion_criteria,patient_summary,patient_exclusion_evidence,trial_title,trial_brief_summary,trial_eligibility_criteria,clinical_space_number,matched_clinical_space_summary,extracted_trial_exclusion_criteria
0,11609,NCT06810544,1,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,"A Phase 1/2, Multicenter, Open-Label Study to ...",This is a first in human study of TNG456 alone...,Inclusion Criteria:\n\n* Has a tumor with a co...,3,Age range allowed: >=18 years. Sex allowed: Bo...,Pregnant or breastfeeding female patients.\nIm...
1,11609,NCT06552260,2,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,A Surgical Window of Opportunity Clinical Tria...,This research study is studying troriluzole as...,Inclusion Criteria:\n\n* Age ≥18 years\n* Hist...,1,Age range allowed: >=18 years. Sex allowed: Bo...,Poor performance status (Karnofsky Performance...
2,11609,NCT06287463,3,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,"A Master Protocol for the Multi-Cohort, Open-L...","This is a multicenter, Phase 1/2 clinical tria...",Inclusion Criteria:\n\nGeneral Inclusion Crite...,6,Age range allowed: NA. Sex allowed: Both. Canc...,"Poor performance status (e.g., ECOG > 1)\nPreg..."
3,11609,NCT04557449,4,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,"A PHASE 1/2A STUDY EVALUATING THE SAFETY, TOLE...","This is a Phase 1/2A, open label, multicenter,...",Inclusion Criteria\n\n* Part 1: Breast Cancer ...,8,Age range allowed: NA. Sex allowed: Both. Canc...,Poor performance status (eg ECOG >1)\nRenal dy...
4,11609,NCT03423628,5,False,Age: 71\nSex: Female\nCancer type: Glioblastom...,ECOG 4. Essential hypertension (grade 3 during...,"A Phase I, Multicentre Study to Assess the Saf...",This study will test an investigational drug c...,Inclusion Criteria:\n\n* Provision of formalin...,1,Age range allowed: NA. Sex allowed: Both. Canc...,History of severe brain-injury or stroke.\nIne...


In [64]:
# Optional local export for review or downstream analysis.
final_matches.to_csv("output/final_matches.csv", index=None)